# Face verification on CPU
This notebook compares two uploaded videos with pretrained RetinaFace and ArcFace models. No training or GPU is required. Run the cells in order; the first model run downloads `buffalo_l`.

In [ ]:
from pathlib import Path

repo = Path('/content/video-face-verification')
if not repo.exists():
    !git clone https://github.com/soheibkhodamoradi/video-face-verification.git {repo}
%cd {repo}
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e . --no-deps

In [ ]:
from google.colab import files

print('Choose exactly two video files.')
uploaded = files.upload()
if len(uploaded) != 2:
    raise ValueError(f'Expected two videos, received {len(uploaded)} files.')
video1, video2 = list(uploaded.keys())
print(f'First video: {video1}\nSecond video: {video2}')

In [ ]:
from face_verify.verify import FaceVerifier

# Adjust these values for your own validated operating point.
strategy = 'mean'  # or 'pairwise'
threshold = 0.4
every = 5

verifier = FaceVerifier(every=every)
result = verifier.verify(video1, video2, strategy=strategy, threshold=threshold)
print(f'Similarity: {result.similarity:.6f}')
print(f'Decision: {result.decision} (threshold {threshold})')
print(f'Usable frames: {result.usable_frames_video1}, {result.usable_frames_video2}')

In [ ]:
import matplotlib.pyplot as plt
from face_verify.frames import iter_sampled_frames

def show_first_accepted_face(video_path, title):
    for frame in iter_sampled_frames(video_path, every=every):
        face = verifier.detector.detect(frame.image, frame.index)
        if face is None:
            continue
        h, w = frame.image.shape[:2]
        x1, y1, x2, y2 = face.bbox.astype(int)
        crop = frame.image[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
        plt.figure(figsize=(3, 3))
        plt.imshow(crop[:, :, ::-1])  # OpenCV BGR -> matplotlib RGB
        plt.title(f'{title}\nframe {frame.index}, blur {face.blur_score:.1f}')
        plt.axis('off')
        plt.show()
        return
    raise RuntimeError(f'No accepted face to display in {video_path}')

show_first_accepted_face(video1, 'Video 1')
show_first_accepted_face(video2, 'Video 2')

The displayed crops confirm which faces were used. The default threshold is only a starting point: collect labeled pairs and run `evaluate.py` to select and validate a threshold for your data.